# National watchlist score distribution maps — England & Wales

This notebook visualises the national distribution of `initial_watchlist_score` across England and Wales using WD25 ward boundaries.

It creates:

- an England & Wales ward-level score-band map;
- separate England and Wales maps;
- one score-band map per English region plus Wales;
- a regional median-score map;
- a high-score concentration map;
- regional summary tables for medians, quartiles, score bands and top wards.

The notebook assumes your model file already contains WD25 ward codes and region fields. It does **not** need the MSOA lookup file unless you want to do separate MSOA-level mapping later.

## Print-quality readability update

This version further improves the map outputs:

- fixes the overlapping statistics block by drawing stats as explicit key/value rows
- reduces ward-border clutter by suppressing internal ward outlines by default
- adds total population to the summary block and population totals to each score-band legend row
- adds a thin red outline around wards where the SDP has previously stood, where this can be identified from the watchlist or optional SDP results files
- increases PNG output size/DPI and can also export SVG/PDF vector copies for print or zooming


## 1. Setup and paths

Expected project structure:

```text
Electoral_Tribes/
  data/
    processed/
      initial_watchlist_scores_ward25_all_available_v2.csv
    geography/
      boundaries/
        <WD25 ward boundaries for England and Wales>.gpkg/.shp/.geojson
  notebooks/
```

The notebook also searches the current directory, project root, `data/`, `data/processed/`, `data/geography/`, `data/geography/boundaries/`, and `/mnt/data`.

Boundary file requirement: a WD25 ward boundary file containing a `WD25CD` column and polygon geometry. If your boundary file has a different code column name, update `WARD_BOUNDARY_CODE_CANDIDATES` in the configuration cell.

In [6]:
from pathlib import Path
from datetime import datetime
import warnings
import textwrap
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle

warnings.filterwarnings("ignore")

try:
    import geopandas as gpd
except Exception as e:
    raise ImportError(
        "geopandas is required. In your environment, install it with something like: "
        "pip install geopandas matplotlib pandas pyogrio mapclassify openpyxl"
    ) from e

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
BOUNDARY_DIR = GEOGRAPHY_DIR

INPUT_DIRS = [
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR / "report_assets_pre_adam_v1" / "tables",
    PROCESSED_DIR / "report_assets_pre_adam_v1" / "appendices",
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "sdp_campaign_validation_v2",
    PROCESSED_DIR / "sdp_campaign_validation_v1",
    PROCESSED_DIR,
    DATA_DIR,
    BOUNDARY_DIR,
    GEOGRAPHY_DIR,
    NOTEBOOK_DIR,
    PROJECT_DIR,
    Path("/mnt/data"),
]

OUTPUT_DIR = PROCESSED_DIR / "national_score_distribution_maps_v2_print_quality"
MAP_DIR = OUTPUT_DIR / "maps"
TABLE_DIR = OUTPUT_DIR / "tables"
MANIFEST_DIR = OUTPUT_DIR / "manifest"
for p in [OUTPUT_DIR, MAP_DIR, TABLE_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

manifest_rows = []
print("Output directory:", OUTPUT_DIR)


Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality


## 2. Configuration

The score is treated as a 0–100 value. If a score column appears to be 0–1, it is automatically scaled to 0–100.

The map colours are deliberately muted. The aim is to show opportunity gradients without making mid-ranking wards look visually negative.

In [7]:
# -----------------------------
# Input file candidates
# -----------------------------
MODEL_CANDIDATES = [
    "initial_watchlist_scores_ward25_all_available_v2.csv",
    "EDITED-initial_watchlist_scores_ward25_all_available_v2.csv",
    "target_score_components_ward25_all_available_v2.csv",
    "all_available_consolidated_review_full_v1.csv",
    "all_available_consolidated_target_review_v1.csv",
]

SCORE_COLUMN_CANDIDATES = [
    "initial_watchlist_score",
    "structural_opportunity_score",
    "model_score",
    "target_score",
    "score",
]

POPULATION_COLUMN_CANDIDATES = [
    "population",
    "total_population",
    "usual_residents",
    "allocated_population",
    "ward_population",
]

WARD_CODE_CANDIDATES = ["WD25CD", "WD25CD_model", "WD25CD_final", "ward_code"]
WARD_NAME_CANDIDATES = ["WD25NM", "WD25NM_model", "WD25NM_final", "ward_name"]
LAD_CODE_CANDIDATES = ["LAD25CD", "LAD25CD_model", "LAD25CD_final", "lad_code"]
LAD_NAME_CANDIDATES = ["LAD25NM", "LAD25NM_model", "LAD25NM_final", "lad_name"]
REGION_NAME_CANDIDATES = ["analysis_region", "RGN25NM", "region", "region_name"]
COUNTRY_CANDIDATES = ["country_inferred", "country", "country_name"]
WARD_BOUNDARY_CODE_CANDIDATES = ["WD25CD", "WD24CD", "WD23CD", "ward_code", "WDCD"]

# Optional explicit boundary path. Leave as None to auto-search.
BOUNDARY_FILE_OVERRIDE = None

# Optional SDP result sources. The notebook uses these to draw red borders around previously SDP-contested wards.
# It first uses any SDP vote fields already in the watchlist, then optionally supplements this from these files.
SDP_VALIDATION_CSV_CANDIDATES = [
    "sdp_campaign_wards_profile_v2.csv",
    "sdp_campaign_wards_profile_v1.csv",
    "appendix_sdp_highest_vote_share_cases_v1.csv",
    "appendix_yorkshire_sdp_case_study_wards_v1.csv",
]
SDP_RESULTS_XLSX_CANDIDATES = [
    "SDP Election Results since 2018.xlsx",
]
SDP_RESULTS_SHEETS = ["WardsEDs", "(Wards) South Yorkshire Mayoral"]

# Geometry simplification. Set to 0 for maximum detail. 10–25 is usually fine for large print PNGs.
SIMPLIFY_TOLERANCE_METRES = 10
TARGET_CRS = "EPSG:27700"  # British National Grid

# -----------------------------
# Map styling
# -----------------------------
BACKGROUND = "#FFFFFF"
OTHER_WARD_FILL = "#F2F2F2"
NO_SCORE_FILL = "#E0E0E0"
BASE_EDGE = "#D6D6D6"
REGION_OUTLINE = "#333333"
LAD_OUTLINE = "#4D4D4D"
TEXT_DARK = "#111111"
TEXT_MID = "#4A4A4A"

# Red border marks wards where the SDP has previously stood.
SDP_BORDER_COLOR = "#D7191C"
SDP_BORDER_WIDTH = 0.75
SHOW_SDP_BORDERS = True

# Border controls. Internal ward borders create heavy visual noise at national scale, so they are off by default.
SHOW_INTERNAL_WARD_BORDERS = False
WARD_BORDER_WIDTH = 0.03
SHOW_LAD_BOUNDARIES = True
LAD_BOUNDARY_WIDTH = 0.18
SELECTED_AREA_OUTLINE_WIDTH = 0.70

SCORE_BANDS = [
    {"label": "Below 40", "lower": -np.inf, "upper": 40, "color": "#D9D9D9"},
    {"label": "40–44.9", "lower": 40, "upper": 45, "color": "#F5F3D7"},
    {"label": "45–49.9", "lower": 45, "upper": 50, "color": "#ECE7A7"},
    {"label": "50–54.9", "lower": 50, "upper": 55, "color": "#D8D982"},
    {"label": "55–59.9", "lower": 55, "upper": 60, "color": "#BDD17A"},
    {"label": "60–64.9", "lower": 60, "upper": 65, "color": "#91C478"},
    {"label": "65–69.9", "lower": 65, "upper": 70, "color": "#5DAE6A"},
    {"label": "70+", "lower": 70, "upper": np.inf, "color": "#247A4A"},
]
SCORE_BAND_ORDER = [b["label"] for b in SCORE_BANDS]
SCORE_BAND_COLORS = {b["label"]: b["color"] for b in SCORE_BANDS}
SCORE_BAND_COLORS["No score"] = NO_SCORE_FILL

# Layout tuning.
DEFAULT_LAYOUT = {
    "figsize": (26.0, 17.0),
    "legend_width_ratio": 2.15,
    "map_width_ratio": 5.20,
    "wspace": 0.025,
    "dpi": 450,
}

# Vector exports are the right option for print or deep zoom. They can be large/slow with ward polygons.
EXPORT_VECTOR = True
VECTOR_FORMATS = ["svg", "pdf"]

TITLE_WRAP = 28
SUBTITLE_WRAP = 46
NOTE_WRAP = 44
FOCUS_PAD = 0.035
COUNTRY_PAD = 0.030
NATIONAL_PAD = 0.012
SHOW_CONTEXT_FOR_COUNTRIES = False
SHOW_CONTEXT_FOR_REGIONS = False

REGION_LABEL_OFFSETS = {
    "North West": (-35000, 18000),
    "Yorkshire and The Humber": (45000, -5000),
    "South East": (25000, -10000),
    "London": (0, -15000),
}

# Development switch. Use this while tuning to avoid regenerating every map.
TEST_MODE = False
TEST_REGION = "North East"  # Example: "North East", "East Midlands", "Wales"


## 3. Utility functions

In [8]:
def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("&", "and")
    for ch in [".", ",", "'", "’", "-", "/", "(", ")"]:
        s = s.replace(ch, " ")
    return " ".join(s.split())


def slugify(value):
    s = str(value).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_") or "unnamed"


def standardise_code(series):
    return series.astype("string").str.strip()


def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def find_file(filename, required=True):
    for folder in INPUT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    # Slow recursive fallback in key project folders only.
    for folder in [PROCESSED_DIR, DATA_DIR, GEOGRAPHY_DIR, PROJECT_DIR]:
        if folder.exists():
            matches = sorted(folder.rglob(filename), key=lambda p: p.stat().st_mtime, reverse=True)
            if matches:
                return matches[0]
    if required:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional missing:", filename)
        return None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df


def score_to_band(value):
    if pd.isna(value):
        return "No score"
    v = float(value)
    for band in SCORE_BANDS:
        if v >= band["lower"] and v < band["upper"]:
            return band["label"]
    return "No score"


def fmt_int(value):
    if pd.isna(value):
        return "n/a"
    return f"{int(round(float(value))):,}"


def fmt_score(value):
    if pd.isna(value):
        return "n/a"
    return f"{float(value):.2f}"


def add_manifest(filename, asset_type, description, path):
    manifest_rows.append({
        "filename": filename,
        "asset_type": asset_type,
        "description": description,
        "path": str(path),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })


def save_table(df, filename, description):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    add_manifest(filename, "table_csv", description, path)
    print("Saved table:", path)
    return path


def save_map(fig, filename, description, dpi=None):
    path = MAP_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor=BACKGROUND, dpi=dpi or DEFAULT_LAYOUT["dpi"])
    add_manifest(filename, "map_png", description, path)
    print("Saved map:", path)

    if EXPORT_VECTOR:
        stem = Path(filename).stem
        for fmt in VECTOR_FORMATS:
            vector_path = MAP_DIR / f"{stem}.{fmt}"
            fig.savefig(vector_path, bbox_inches="tight", facecolor=BACKGROUND, format=fmt)
            add_manifest(vector_path.name, f"map_{fmt}", description + f" ({fmt.upper()} vector export).", vector_path)
            print("Saved vector map:", vector_path)

    plt.close(fig)
    return path


def require_columns(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{label} missing required columns: {missing}. Available columns: {df.columns.tolist()}")


def load_optional_sdp_validation_sources():
    """Return a small dataframe of previously SDP-contested wards where a code or name/authority pair can be found."""
    frames = []

    # Optional CSV validation/profile files, usually already mapped to WD25.
    for fname in SDP_VALIDATION_CSV_CANDIDATES:
        df = read_csv(fname, required=False)
        if df is None or df.empty:
            continue
        code_col = first_existing_col(df, WARD_CODE_CANDIDATES)
        ward_col = first_existing_col(df, WARD_NAME_CANDIDATES + ["ward", "Ward", "Area Name"])
        lad_col = first_existing_col(df, LAD_NAME_CANDIDATES + ["council", "Council", "Authority Name"])
        tmp = pd.DataFrame()
        if code_col:
            tmp["WD25CD"] = standardise_code(df[code_col])
        else:
            tmp["WD25CD"] = pd.NA
        tmp["clean_ward"] = df[ward_col].map(clean_text) if ward_col else ""
        tmp["clean_lad"] = df[lad_col].map(clean_text) if lad_col else ""
        tmp["sdp_source"] = fname
        frames.append(tmp)

    # Optional maintained SDP election workbook. This often has names rather than WD25 codes.
    for fname in SDP_RESULTS_XLSX_CANDIDATES:
        path = find_file(fname, required=False)
        if path is None:
            print("Optional missing:", fname)
            continue
        try:
            xl = pd.ExcelFile(path)
            for sheet in SDP_RESULTS_SHEETS:
                if sheet not in xl.sheet_names:
                    continue
                df = pd.read_excel(path, sheet_name=sheet)
                if df.empty:
                    continue
                if "Area Type" in df.columns:
                    df = df[df["Area Type"].astype(str).str.lower().str.contains("ward", na=False)].copy()
                tmp = pd.DataFrame()
                if "Area Code" in df.columns:
                    tmp["WD25CD"] = standardise_code(df["Area Code"])
                    tmp.loc[~tmp["WD25CD"].astype(str).str.match(r"^[EW]\d{8}$", na=False), "WD25CD"] = pd.NA
                else:
                    tmp["WD25CD"] = pd.NA
                tmp["clean_ward"] = df["Area Name"].map(clean_text) if "Area Name" in df.columns else ""
                tmp["clean_lad"] = df["Authority Name"].map(clean_text) if "Authority Name" in df.columns else ""
                tmp["sdp_source"] = f"{fname}::{sheet}"
                frames.append(tmp)
        except Exception as e:
            print(f"Could not read optional SDP workbook {fname}: {e}")

    if not frames:
        return pd.DataFrame(columns=["WD25CD", "clean_ward", "clean_lad", "sdp_source"])

    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates()
    out = out[(out["WD25CD"].notna()) | ((out["clean_ward"] != "") & (out["clean_lad"] != ""))].copy()
    return out


## 4. Load and prepare the watchlist model

In [9]:
model = None
model_source = None

for fname in MODEL_CANDIDATES:
    df = read_csv(fname, required=False)
    if df is None or df.empty:
        continue
    if first_existing_col(df, WARD_CODE_CANDIDATES) and first_existing_col(df, SCORE_COLUMN_CANDIDATES):
        model = df.copy()
        model_source = fname
        break

if model is None:
    raise FileNotFoundError(
        "Could not find a usable model file. Expected a CSV with a WD25 ward code column and an initial/model score column."
    )

print("Using model source:", model_source)
print("Rows:", len(model))
print("Columns:", len(model.columns))

# Standardise core columns.
ward_code_col = first_existing_col(model, WARD_CODE_CANDIDATES)
ward_name_col = first_existing_col(model, WARD_NAME_CANDIDATES)
lad_code_col = first_existing_col(model, LAD_CODE_CANDIDATES)
lad_name_col = first_existing_col(model, LAD_NAME_CANDIDATES)
region_col = first_existing_col(model, REGION_NAME_CANDIDATES)
country_col = first_existing_col(model, COUNTRY_CANDIDATES)
score_col = first_existing_col(model, SCORE_COLUMN_CANDIDATES)
pop_col = first_existing_col(model, POPULATION_COLUMN_CANDIDATES)

required_found = {
    "ward_code": ward_code_col,
    "ward_name": ward_name_col,
    "lad_code": lad_code_col,
    "lad_name": lad_name_col,
    "region": region_col,
    "score": score_col,
    "population": pop_col,
}
print("Resolved columns:", required_found)

if ward_code_col is None or score_col is None:
    raise KeyError("Model must contain a ward code column and a score column.")

model["WD25CD"] = standardise_code(model[ward_code_col])
model["WD25NM"] = model[ward_name_col] if ward_name_col else ""
model["LAD25CD"] = model[lad_code_col] if lad_code_col else ""
model["LAD25NM"] = model[lad_name_col] if lad_name_col else ""
model["analysis_region"] = model[region_col] if region_col else "Unknown"
model["clean_ward"] = model["WD25NM"].map(clean_text)
model["clean_lad"] = model["LAD25NM"].map(clean_text)

if country_col:
    model["country_inferred"] = model[country_col]
else:
    # WD25 codes beginning E are England; W are Wales.
    model["country_inferred"] = np.where(model["WD25CD"].str.startswith("W"), "Wales", "England")

model["score_for_map"] = pd.to_numeric(model[score_col], errors="coerce")
if model["score_for_map"].dropna().max() <= 1.5:
    model["score_for_map"] = model["score_for_map"] * 100

if pop_col:
    model["population_for_map"] = pd.to_numeric(model[pop_col], errors="coerce")
else:
    model["population_for_map"] = np.nan
    print("WARNING: no population column found. Population totals will show as n/a.")

# Keep the England/Wales all-available scope where the field exists.
if "scope_all_available" in model.columns:
    before = len(model)
    model = model[model["scope_all_available"].fillna(False).astype(bool)].copy()
    print(f"Filtered to scope_all_available: {before:,} -> {len(model):,}")

model = model.dropna(subset=["WD25CD", "score_for_map"]).copy()
model["score_band"] = model["score_for_map"].map(score_to_band)
model["score_band"] = pd.Categorical(model["score_band"], categories=SCORE_BAND_ORDER + ["No score"], ordered=True)
model["score_band_color"] = model["score_band"].astype(str).map(SCORE_BAND_COLORS).fillna(OTHER_WARD_FILL)

# SDP-contested flag from the watchlist itself, plus optional validation/election files.
model["sdp_contested"] = False
for c in ["latest_election_allocated_sdp_votes", "latest_election_sdp_votes", "sdp_votes", "Votes"]:
    if c in model.columns:
        model["sdp_contested"] = model["sdp_contested"] | (pd.to_numeric(model[c], errors="coerce").fillna(0) > 0)
for c in ["latest_election_sdp_share", "sdp_share", "Votes (Perc.)"]:
    if c in model.columns:
        model["sdp_contested"] = model["sdp_contested"] | (pd.to_numeric(model[c], errors="coerce").fillna(0) > 0)

sdp_sources = load_optional_sdp_validation_sources()
if len(sdp_sources):
    sdp_codes = set(sdp_sources["WD25CD"].dropna().astype(str))
    if sdp_codes:
        model["sdp_contested"] = model["sdp_contested"] | model["WD25CD"].isin(sdp_codes)

    name_match = sdp_sources[sdp_sources["WD25CD"].isna() & (sdp_sources["clean_ward"] != "") & (sdp_sources["clean_lad"] != "")]
    if len(name_match):
        name_keys = set(zip(name_match["clean_lad"], name_match["clean_ward"]))
        name_hits = pd.Series(
    [(lad, ward) in name_keys for lad, ward in zip(model["clean_lad"], model["clean_ward"])],
    index=model.index,
    dtype=bool,
)

model["sdp_contested"] = model["sdp_contested"].fillna(False).astype(bool) | name_hits

print("Prepared model rows:", len(model))
print("Score column:", score_col)
print("Population column:", pop_col)
print("SDP-contested wards flagged:", int(model["sdp_contested"].sum()))
print(model[["WD25CD", "WD25NM", "LAD25NM", "analysis_region", "country_inferred", "population_for_map", "score_for_map", "score_band", "sdp_contested"]].head())


Loaded initial_watchlist_scores_ward25_all_available_v2.csv: (7572, 142) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\outputs\initial_watchlist_scores_ward25_all_available_v2.csv
Using model source: initial_watchlist_scores_ward25_all_available_v2.csv
Rows: 7572
Columns: 142
Resolved columns: {'ward_code': 'WD25CD', 'ward_name': 'WD25NM', 'lad_code': 'LAD25CD', 'lad_name': 'LAD25NM', 'region': 'analysis_region', 'score': 'initial_watchlist_score', 'population': 'population'}
Filtered to scope_all_available: 7,572 -> 7,572
Loaded sdp_campaign_wards_profile_v2.csv: (219, 64) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2\sdp_campaign_wards_profile_v2.csv
Loaded sdp_campaign_wards_profile_v1.csv: (171, 76) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v1\sdp_campaign_wards_profile_v1.csv
Loaded appendix_sdp_highest_vote_share_cases_v1.csv: (100, 15) from c:\Users\keena\Docume

## 5. Regional summary checks

This section gives the exact summary table behind the maps. It is also a useful guardrail against copied-table errors, such as a median appearing above the third quartile.

In [10]:
def q1(x):
    return x.quantile(0.25)


def q3(x):
    return x.quantile(0.75)

region_summary = (
    model.groupby("analysis_region", dropna=False)
    .agg(
        wards=("WD25CD", "count"),
        population=("population_for_map", "sum"),
        min=("score_for_map", "min"),
        q1=("score_for_map", q1),
        median=("score_for_map", "median"),
        q3=("score_for_map", q3),
        max=("score_for_map", "max"),
        mean=("score_for_map", "mean"),
        sd=("score_for_map", "std"),
        sdp_contested_wards=("sdp_contested", "sum"),
    )
    .reset_index()
    .sort_values(["median", "mean"], ascending=False)
)

for col in ["min", "q1", "median", "q3", "max", "mean", "sd"]:
    region_summary[col] = region_summary[col].round(2)
region_summary["population"] = region_summary["population"].round(0).astype("Int64")
region_summary["sdp_contested_wards"] = region_summary["sdp_contested_wards"].astype(int)

# Simple logical check.
bad_quartiles = region_summary[
    ~((region_summary["min"] <= region_summary["q1"]) &
      (region_summary["q1"] <= region_summary["median"]) &
      (region_summary["median"] <= region_summary["q3"]) &
      (region_summary["q3"] <= region_summary["max"]))
]

if len(bad_quartiles):
    print("WARNING: quartile ordering issue detected")
    display(bad_quartiles)
else:
    print("Quartile ordering check passed.")

display(region_summary)
save_table(region_summary, "national_region_score_summary_v2.csv", "Regional summary of initial watchlist score with population and SDP-contested ward counts.")


Quartile ordering check passed.


,analysis_region,wards,population,min,q1,median,q3,max,mean,sd,sdp_contested_wards
0,East Midlands,761,4880068,22.29,55.25,61.63,66.13,81.41,59.99,9.75,8
3,North East,334,2647019,30.71,53.24,60.87,66.54,81.10,59.52,9.56,9
6,South West,821,5701210,28.06,51.99,59.54,66.37,81.99,59.06,9.93,4
7,Wales,762,3107463,23.29,51.46,58.61,64.91,81.14,58.11,9.38,1
8,West Midlands,754,5950764,15.54,49.77,58.14,64.64,80.28,55.82,12.16,4
1,East of England,947,6334922,23.38,49.23,55.89,62.10,76.73,55.18,9.80,1
5,South East,1269,9278116,22.34,47.18,54.25,61.31,81.00,53.93,10.05,12
9,Yorkshire and The Humber,410,5480866,24.23,46.91,53.90,60.58,78.63,53.64,9.87,57
4,North West,825,7417549,21.69,46.69,53.90,60.28,79.76,53.18,10.14,11
2,London,689,8799770,15.79,25.33,31.40,37.33,67.04,32.24,8.82,10


Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\tables\national_region_score_summary_v2.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/tables/national_region_score_summary_v2.csv')

In [11]:
region_band_counts = (
    pd.crosstab(model["analysis_region"], model["score_band"])
    .reindex(columns=SCORE_BAND_ORDER + ["No score"], fill_value=0)
    .reset_index()
)
region_band_counts["total_wards"] = region_band_counts[SCORE_BAND_ORDER].sum(axis=1)
for band in SCORE_BAND_ORDER:
    region_band_counts[f"{band}_share"] = (region_band_counts[band] / region_band_counts["total_wards"]).round(4)

display(region_band_counts)
save_table(region_band_counts, "national_region_score_band_counts_v1.csv", "Score-band counts and shares by region.")

score_band,analysis_region,Below 40,40–44.9,45–49.9,50–54.9,55–59.9,60–64.9,65–69.9,70+,No score,total_wards,Below 40_share,40–44.9_share,45–49.9_share,50–54.9_share,55–59.9_share,60–64.9_share,65–69.9_share,70+_share
0,East Midlands,37,29,43,74,127,219,136,96,0,761,0.0486,0.0381,0.0565,0.0972,0.1669,0.2878,0.1787,0.1261
1,East of England,71,72,118,182,188,169,94,53,0,947,0.0750,0.0760,0.1246,0.1922,0.1985,0.1785,0.0993,0.0560
2,London,573,56,34,8,12,4,2,0,0,689,0.8316,0.0813,0.0493,0.0116,0.0174,0.0058,0.0029,0.0000
3,North East,12,11,34,40,59,75,63,40,0,334,0.0359,0.0329,0.1018,0.1198,0.1766,0.2246,0.1886,0.1198
4,North West,87,82,133,140,164,118,75,26,0,825,0.1055,0.0994,0.1612,0.1697,0.1988,0.1430,0.0909,0.0315
5,South East,114,124,183,255,228,183,127,55,0,1269,0.0898,0.0977,0.1442,0.2009,0.1797,0.1442,0.1001,0.0433
6,South West,23,58,82,114,145,143,141,115,0,821,0.0280,0.0706,0.0999,0.1389,0.1766,0.1742,0.1717,0.1401
7,Wales,23,29,103,132,139,147,124,65,0,762,0.0302,0.0381,0.1352,0.1732,0.1824,0.1929,0.1627,0.0853
8,West Midlands,87,47,59,111,113,159,115,63,0,754,0.1154,0.0623,0.0782,0.1472,0.1499,0.2109,0.1525,0.0836
9,Yorkshire and The Humber,32,49,64,81,74,61,30,19,0,410,0.0780,0.1195,0.1561,0.1976,0.1805,0.1488,0.0732,0.0463


Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\tables\national_region_score_band_counts_v1.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/tables/national_region_score_band_counts_v1.csv')

In [12]:
# Top wards by region for interpretation and QA.
TOP_N_PER_REGION = 25

top_wards_by_region = (
    model.sort_values(["analysis_region", "score_for_map"], ascending=[True, False])
    .groupby("analysis_region", group_keys=False)
    .head(TOP_N_PER_REGION)
    [["analysis_region", "country_inferred", "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "score_for_map", "score_band"]]
    .sort_values(["analysis_region", "score_for_map"], ascending=[True, False])
)

top_wards_by_region["score_for_map"] = top_wards_by_region["score_for_map"].round(2)
display(top_wards_by_region.head(50))
save_table(top_wards_by_region, "national_top_wards_by_region_v1.csv", f"Top {TOP_N_PER_REGION} wards by initial watchlist score within each region.")

,analysis_region,country_inferred,LAD25CD,LAD25NM,WD25CD,WD25NM,score_for_map,score_band
3787,East Midlands,England,E07000137,East Lindsey,E05009903,Trinity,81.41,70+
3861,East Midlands,England,E07000141,South Kesteven,E05010162,Grantham Harrowby,80.77,70+
3881,East Midlands,England,E07000142,West Lindsey,E05009640,Gainsborough East,80.24,70+
2148,East Midlands,England,E07000037,High Peak,E05010632,Gamesley,80.23,70+
3883,East Midlands,England,E07000142,West Lindsey,E05009642,Gainsborough South-West,79.75,70+
3882,East Midlands,England,E07000142,West Lindsey,E05009641,Gainsborough North,79.12,70+
4088,East Midlands,England,E07000170,Ashfield,E05010687,Leamington,79.01,70+
4077,East Midlands,England,E07000170,Ashfield,E05010676,Carsic,78.82,70+
2093,East Midlands,England,E07000034,Chesterfield,E05015180,Staveley Central,78.78,70+
1661,East Midlands,England,E06000061,North Northamptonshire,E05016203,Avondale Grange,78.40,70+


Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\tables\national_top_wards_by_region_v1.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/tables/national_top_wards_by_region_v1.csv')

## 6. Load WD25 ward boundaries

If this cell fails, place a WD25 ward boundary file in one of the searched folders. The file must contain ward geometry and a ward code column matching the model's `WD25CD` values.

In [13]:
def find_boundary_file():
    if BOUNDARY_FILE_OVERRIDE:
        p = Path(BOUNDARY_FILE_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(f"BOUNDARY_FILE_OVERRIDE does not exist: {p}")
        return p

    search_dirs = [BOUNDARY_DIR, GEOGRAPHY_DIR, DATA_DIR, PROJECT_DIR, Path("/mnt/data")]
    patterns = ["*.gpkg", "*.shp", "*.geojson", "*.json"]
    candidates = []
    for folder in search_dirs:
        if folder.exists():
            for pattern in patterns:
                candidates.extend(folder.rglob(pattern))

    if not candidates:
        return None

    def score_candidate(p):
        name = p.name.lower()
        score = 0
        for token in ["wd25", "ward", "wards", "dec_2025", "may_2025", "bfc", "bgc", "ew"]:
            if token in name:
                score += 10
        for token in ["oa", "lsoa", "msoa", "lad", "region"]:
            if token in name:
                score -= 5
        # Prefer geopackage/geojson over shapefile where names tie.
        if p.suffix.lower() == ".gpkg":
            score += 2
        if p.suffix.lower() in [".geojson", ".json"]:
            score += 1
        return score

    candidates = sorted(candidates, key=lambda p: (score_candidate(p), p.stat().st_mtime), reverse=True)
    return candidates[0]

boundary_file = find_boundary_file()
print("Boundary file:", boundary_file)

if boundary_file is None:
    raise FileNotFoundError(
        "No boundary file found. Add a WD25 ward boundary .gpkg/.shp/.geojson to data/geography/boundaries/ "
        "or set BOUNDARY_FILE_OVERRIDE."
    )

wards = gpd.read_file(boundary_file)
print("Boundary rows:", len(wards))
print("Boundary columns:", wards.columns.tolist())

boundary_code_col = first_existing_col(wards, WARD_BOUNDARY_CODE_CANDIDATES)
if boundary_code_col is None:
    raise KeyError(
        "Could not find a ward-code column in the boundary file. "
        f"Tried: {WARD_BOUNDARY_CODE_CANDIDATES}. Available: {wards.columns.tolist()}"
    )

wards["WD25CD"] = standardise_code(wards[boundary_code_col])

if wards.crs is None:
    print("Boundary CRS is missing. Setting CRS to EPSG:27700. Check this manually if your file uses another CRS.")
    wards = wards.set_crs(TARGET_CRS)
elif str(wards.crs).upper() != TARGET_CRS:
    wards = wards.to_crs(TARGET_CRS)

# Clean invalid geometries where possible.
try:
    invalid_count = int((~wards.geometry.is_valid).sum())
    if invalid_count:
        print(f"Repairing {invalid_count} invalid geometries with buffer(0).")
        wards["geometry"] = wards.geometry.buffer(0)
except Exception as e:
    print("Geometry validity check skipped:", e)

if SIMPLIFY_TOLERANCE_METRES and SIMPLIFY_TOLERANCE_METRES > 0:
    wards["geometry"] = wards.geometry.simplify(SIMPLIFY_TOLERANCE_METRES, preserve_topology=True)

wards = wards[["WD25CD", "geometry"]].drop_duplicates("WD25CD").copy()
print("Prepared boundary rows:", len(wards))

Boundary file: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg
Boundary rows: 8405
Boundary columns: ['WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'LAD25NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']
Prepared boundary rows: 8405


## 7. Join scores to boundaries

In [14]:
map_cols = [
    "WD25CD", "WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "country_inferred",
    "population_for_map", "score_for_map", "score_band", "score_band_color", "sdp_contested"
]
model_for_map = model[map_cols].drop_duplicates("WD25CD").copy()

# Prevent pandas from creating _x/_y suffixes where the boundary file already carries name/code fields.
# The model table is the authoritative source for reporting columns; the boundary file supplies geometry.
overlap_cols = [
    c for c in ["WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "country_inferred"]
    if c in wards.columns
]
wards_for_merge = wards.drop(columns=overlap_cols, errors="ignore").copy()

map_gdf = wards_for_merge.merge(model_for_map, on="WD25CD", how="inner")

if len(map_gdf) == 0:
    raise ValueError("No boundary rows matched the model WD25CD codes. Check that the boundary file is WD25 and covers England/Wales.")

for required_col in ["WD25NM", "LAD25CD", "LAD25NM", "analysis_region", "country_inferred", "population_for_map", "score_for_map", "score_band", "sdp_contested"]:
    if required_col not in map_gdf.columns:
        raise KeyError(f"Joined map layer missing required column: {required_col}. Available columns: {map_gdf.columns.tolist()}")

unmapped_model = model_for_map[~model_for_map["WD25CD"].isin(map_gdf["WD25CD"])]
unmapped_boundary = wards[~wards["WD25CD"].isin(model_for_map["WD25CD"])]

print(f"Mapped wards: {len(map_gdf):,}")
print(f"Mapped population: {fmt_int(map_gdf['population_for_map'].sum())}")
print(f"Mapped SDP-contested wards: {int(map_gdf['sdp_contested'].sum()):,}")
print(f"Model rows without boundary match: {len(unmapped_model):,}")
print(f"Boundary rows without model match: {len(unmapped_boundary):,}")
print(map_gdf.groupby("analysis_region").size().sort_values(ascending=False))

save_table(map_gdf.drop(columns="geometry"), "national_score_map_ward_index_v2.csv", "Ward-level map index after joining scores, population and SDP-contested flags to WD25 ward boundaries.")
if len(unmapped_model):
    save_table(unmapped_model, "national_unmapped_model_wards_v2.csv", "Model wards not matched to the boundary file.")


Mapped wards: 7,572
Mapped population: 59,597,747
Mapped SDP-contested wards: 117
Model rows without boundary match: 0
Boundary rows without model match: 833
analysis_region
South East                  1269
East of England              947
North West                   825
South West                   821
Wales                        762
East Midlands                761
West Midlands                754
London                       689
Yorkshire and The Humber     410
North East                   334
dtype: int64
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\tables\national_score_map_ward_index_v2.csv


## 8. Map plotting functions

In [15]:
def wrap_lines(text, width=36):
    if text is None:
        return []
    text = str(text).strip()
    if not text:
        return []
    lines = []
    for para in text.split("\n"):
        para = para.strip()
        if not para:
            lines.append("")
        else:
            lines.extend(textwrap.wrap(para, width=width, break_long_words=False, replace_whitespace=True))
    return lines


def draw_wrapped_block(ax, x, y_top, text, width=36, fontsize=10, fontweight="normal", color=TEXT_DARK, line_gap=1.45):
    lines = wrap_lines(text, width=width)
    if not lines:
        return y_top
    step = 0.030 * (fontsize / 11.0) * line_gap
    y = y_top
    for line in lines:
        ax.text(x, y, line, ha="left", va="top", fontsize=fontsize, fontweight=fontweight, color=color)
        y -= step
    return y


def draw_stat_block(ax, x, y_top, gdf, fontsize=12.0, row_gap=0.034):
    s = gdf["score_for_map"].dropna()
    pop = gdf["population_for_map"].sum(min_count=1) if "population_for_map" in gdf.columns else np.nan
    stat_rows = [
        ("Mapped wards", fmt_int(len(gdf))),
        ("Population", fmt_int(pop)),
        ("Median", fmt_score(s.median()) if len(s) else "n/a"),
        ("Mean", fmt_score(s.mean()) if len(s) else "n/a"),
        ("IQR", f"{fmt_score(s.quantile(0.25))}–{fmt_score(s.quantile(0.75))}" if len(s) else "n/a"),
        ("Range", f"{fmt_score(s.min())}–{fmt_score(s.max())}" if len(s) else "n/a"),
    ]
    if "sdp_contested" in gdf.columns:
        stat_rows.append(("SDP stood", fmt_int(gdf["sdp_contested"].fillna(False).astype(bool).sum())))

    y = y_top
    label_x = x
    value_x = x + 0.47
    for label, value in stat_rows:
        ax.text(label_x, y, f"{label}:", ha="left", va="top", fontsize=fontsize, color=TEXT_DARK)
        ax.text(value_x, y, value, ha="left", va="top", fontsize=fontsize, color=TEXT_DARK)
        y -= row_gap
    return y


def set_padded_bounds(ax, gdf, pad_frac=0.04):
    xmin, ymin, xmax, ymax = gdf.total_bounds
    x_span = max(xmax - xmin, 1)
    y_span = max(ymax - ymin, 1)
    x_pad = x_span * pad_frac
    y_pad = y_span * pad_frac
    ax.set_xlim(xmin - x_pad, xmax + x_pad)
    ax.set_ylim(ymin - y_pad, ymax + y_pad)


def draw_score_band_key(ax, gdf, x=0.0, y_top=0.54, box_size=0.022, row_gap=0.044, fontsize=10.6):
    ax.text(x, y_top + 0.045, "Initial watchlist score", ha="left", va="top", fontsize=12.5, fontweight="bold", color=TEXT_DARK)
    y = y_top
    for band in SCORE_BAND_ORDER:
        band_mask = gdf["score_band"].astype(str).eq(band)
        count = int(band_mask.sum())
        band_pop = gdf.loc[band_mask, "population_for_map"].sum(min_count=1) if "population_for_map" in gdf.columns else np.nan
        rect = Rectangle((x, y - box_size), box_size, box_size, facecolor=SCORE_BAND_COLORS[band], edgecolor=BASE_EDGE, linewidth=0.5)
        ax.add_patch(rect)
        # Two-part row avoids a very long legend line and improves readability.
        ax.text(x + box_size + 0.018, y + 0.002, f"{band}  ({count:,})", ha="left", va="top", fontsize=fontsize, color=TEXT_DARK)
        ax.text(x + box_size + 0.018, y - 0.021, f"Pop: {fmt_int(band_pop)}", ha="left", va="top", fontsize=fontsize * 0.88, color=TEXT_MID)
        y -= row_gap
    return y


def plot_score_band_map(
    gdf,
    title,
    subtitle,
    filename,
    description,
    context_gdf=None,
    figsize=None,
    legend_width_ratio=None,
    map_width_ratio=None,
    wspace=None,
    show_context=False,
    focus_pad=FOCUS_PAD,
    note_text=None,
):
    if len(gdf) == 0:
        print("Skipping empty map:", title)
        return None

    figsize = figsize or DEFAULT_LAYOUT["figsize"]
    legend_width_ratio = legend_width_ratio if legend_width_ratio is not None else DEFAULT_LAYOUT["legend_width_ratio"]
    map_width_ratio = map_width_ratio if map_width_ratio is not None else DEFAULT_LAYOUT["map_width_ratio"]
    wspace = wspace if wspace is not None else DEFAULT_LAYOUT["wspace"]
    note_text = note_text or "Note: score bands show structural opportunity, not vote-share forecasts or target-seat decisions. Red outlines mark previously SDP-contested wards where identified."

    plot_gdf = gdf.copy()
    plot_gdf["score_band"] = plot_gdf["score_band"].astype("string").fillna("No score").astype(str)
    plot_gdf["score_band"] = pd.Categorical(plot_gdf["score_band"], categories=SCORE_BAND_ORDER + ["No score"], ordered=True)
    plot_gdf = plot_gdf.sort_values("score_band")

    fig = plt.figure(figsize=figsize)
    fig.patch.set_facecolor(BACKGROUND)
    gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[legend_width_ratio, map_width_ratio], wspace=wspace)

    legend_ax = fig.add_subplot(gs[0, 0])
    map_ax = fig.add_subplot(gs[0, 1])
    legend_ax.set_facecolor(BACKGROUND)
    map_ax.set_facecolor(BACKGROUND)

    legend_ax.set_axis_off()
    legend_ax.set_xlim(0, 1)
    legend_ax.set_ylim(0, 1)

    if show_context and context_gdf is not None and len(context_gdf):
        context_gdf.plot(ax=map_ax, color="#ECECEC", edgecolor="none", linewidth=0)

    # Draw polygons by band without internal ward outlines by default.
    edgecolor = BASE_EDGE if SHOW_INTERNAL_WARD_BORDERS else "none"
    linewidth = WARD_BORDER_WIDTH if SHOW_INTERNAL_WARD_BORDERS else 0

    for band in SCORE_BAND_ORDER:
        sub = plot_gdf[plot_gdf["score_band"].astype(str).eq(band)]
        if len(sub) == 0:
            continue
        sub.plot(ax=map_ax, color=SCORE_BAND_COLORS[band], edgecolor=edgecolor, linewidth=linewidth)

    # No-score/missing polygons, if present.
    no_score = plot_gdf[~plot_gdf["score_band"].astype(str).isin(SCORE_BAND_ORDER)]
    if len(no_score):
        no_score.plot(ax=map_ax, color=NO_SCORE_FILL, edgecolor=edgecolor, linewidth=linewidth)

    # LAD boundaries give structure without the heavy clutter of every ward border.
    if SHOW_LAD_BOUNDARIES and "LAD25CD" in plot_gdf.columns:
        try:
            lad_boundaries = plot_gdf[["LAD25CD", "geometry"]].dissolve(by="LAD25CD")
            lad_boundaries.boundary.plot(ax=map_ax, color=LAD_OUTLINE, linewidth=LAD_BOUNDARY_WIDTH, alpha=0.70)
        except Exception:
            pass

    # SDP-contested border overlay.
    if SHOW_SDP_BORDERS and "sdp_contested" in plot_gdf.columns:
        sdp_sub = plot_gdf[plot_gdf["sdp_contested"].fillna(False).astype(bool)]
        if len(sdp_sub) > 0:
            sdp_sub.boundary.plot(ax=map_ax, color=SDP_BORDER_COLOR, linewidth=SDP_BORDER_WIDTH, alpha=0.95)

    try:
        outline = plot_gdf.dissolve()
        outline.boundary.plot(ax=map_ax, color=REGION_OUTLINE, linewidth=SELECTED_AREA_OUTLINE_WIDTH)
    except Exception:
        pass

    set_padded_bounds(map_ax, plot_gdf, pad_frac=focus_pad)
    map_ax.set_axis_off()
    map_ax.set_aspect("equal")

    # Sidebar.
    y = 0.972
    y = draw_wrapped_block(legend_ax, 0.00, y, title, width=TITLE_WRAP, fontsize=28, fontweight="bold", line_gap=1.08)
    y -= 0.012
    y = draw_wrapped_block(legend_ax, 0.00, y, subtitle, width=SUBTITLE_WRAP, fontsize=14.0, color=TEXT_MID, line_gap=1.20)
    y -= 0.035
    y = draw_stat_block(legend_ax, 0.00, y, plot_gdf, fontsize=14.0, row_gap=0.037)

    legend_y = min(y - 0.030, 0.555)
    bottom_y = draw_score_band_key(legend_ax, plot_gdf, x=0.00, y_top=legend_y, box_size=0.024, row_gap=0.054, fontsize=12.0)

    # SDP border key.
    if SHOW_SDP_BORDERS:
        sdp_y = max(bottom_y - 0.022, 0.165)
        legend_ax.text(0.00, sdp_y, "SDP-contested ward", ha="left", va="top", fontsize=12.2, fontweight="bold", color=TEXT_DARK)
        rect_y = sdp_y - 0.052
        rect = Rectangle((0.00, rect_y), 0.040, 0.030, facecolor="none", edgecolor=SDP_BORDER_COLOR, linewidth=1.6)
        legend_ax.add_patch(rect)
        legend_ax.text(0.055, rect_y + 0.028, "Red outline", ha="left", va="top", fontsize=11.2, color=TEXT_DARK)
        note_y = rect_y - 0.040
    else:
        note_y = max(bottom_y - 0.040, 0.125)

    draw_wrapped_block(legend_ax, 0.00, note_y, note_text, width=NOTE_WRAP, fontsize=10.8, color=TEXT_MID, line_gap=1.25)

    return save_map(fig, filename, description)



## 9. Generate national, country and regional ward-level maps

In [16]:
# Main national map.
plot_score_band_map(
    map_gdf,
    title="England & Wales initial watchlist score",
    subtitle="Ward-level score bands. Higher values indicate stronger structural opportunity under the current model.",
    filename="map_01_england_wales_initial_watchlist_score_bands_v2.png",
    description="England and Wales ward-level initial watchlist score map using fixed 5-point score bands, population totals and SDP-contested borders.",
    figsize=(27.0, 18.0),
    legend_width_ratio=2.20,
    map_width_ratio=5.10,
    wspace=0.025,
    show_context=False,
    focus_pad=NATIONAL_PAD,
)

# Country maps.
for country in sorted(map_gdf["country_inferred"].dropna().unique()):
    sub = map_gdf[map_gdf["country_inferred"].eq(country)].copy()
    plot_score_band_map(
        sub,
        title=f"{country} initial watchlist score",
        subtitle="Ward-level score bands using the same England & Wales scale.",
        filename=f"map_02_country_{slugify(country)}_initial_watchlist_score_bands_v2.png",
        description=f"{country} ward-level initial watchlist score map using fixed score bands, population totals and SDP-contested borders.",
        context_gdf=map_gdf,
        figsize=(26.0, 17.0),
        show_context=SHOW_CONTEXT_FOR_COUNTRIES,
        focus_pad=COUNTRY_PAD,
    )

# Regional maps.
regions = list(region_summary["analysis_region"])
if TEST_MODE:
    regions = [TEST_REGION]
    print("TEST_MODE active. Plotting only:", regions)

for region in regions:
    sub = map_gdf[map_gdf["analysis_region"].eq(region)].copy()
    if len(sub) == 0:
        print("No mapped wards for region:", region)
        continue
    plot_score_band_map(
        sub,
        title=f"{region} initial watchlist score",
        subtitle="Ward-level score bands using the same England & Wales scale.",
        filename=f"map_10_region_{slugify(region)}_initial_watchlist_score_bands_v2.png",
        description=f"{region} ward-level initial watchlist score map using fixed score bands, population totals and SDP-contested borders.",
        context_gdf=map_gdf,
        figsize=(26.0, 17.0),
        show_context=SHOW_CONTEXT_FOR_REGIONS,
        focus_pad=FOCUS_PAD,
    )


Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_01_england_wales_initial_watchlist_score_bands_v2.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_01_england_wales_initial_watchlist_score_bands_v2.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_01_england_wales_initial_watchlist_score_bands_v2.pdf
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_02_country_england_initial_watchlist_score_bands_v2.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_02_country_england_initial_watchlist_score_bands_v2.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\proce

## 10. Regional median-score map

This map dissolves wards into regions and colours each region by its median watchlist score. It is useful for communicating the national pattern without forcing readers to inspect thousands of wards.

In [17]:
region_geom = map_gdf[["analysis_region", "geometry"]].dissolve(by="analysis_region").reset_index()
region_map = region_geom.merge(region_summary, on="analysis_region", how="left")
region_map["median_band"] = region_map["median"].map(score_to_band)
region_map["median_band"] = pd.Categorical(region_map["median_band"], categories=SCORE_BAND_ORDER + ["No score"], ordered=True)

fig = plt.figure(figsize=(26.0, 17.0))
fig.patch.set_facecolor(BACKGROUND)
gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[2.15, 5.20], wspace=0.025)
legend_ax = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[0, 1])
legend_ax.set_facecolor(BACKGROUND)
ax.set_facecolor(BACKGROUND)
legend_ax.set_axis_off()
legend_ax.set_xlim(0, 1)
legend_ax.set_ylim(0, 1)

for band in SCORE_BAND_ORDER:
    sub = region_map[region_map["median_band"].astype(str).eq(band)]
    if len(sub):
        sub.plot(ax=ax, color=SCORE_BAND_COLORS[band], edgecolor=REGION_OUTLINE, linewidth=0.75)

# Add readable labels with light halo.
for _, row in region_map.iterrows():
    try:
        pt = row.geometry.representative_point()
        dx, dy = REGION_LABEL_OFFSETS.get(row["analysis_region"], (0, 0))
        label = f"{row['analysis_region']}\n{row['median']:.1f}"
        ax.text(
            pt.x + dx,
            pt.y + dy,
            label,
            ha="center",
            va="center",
            fontsize=12.0,
            color=TEXT_DARK,
            path_effects=[pe.withStroke(linewidth=4.0, foreground="white")],
        )
    except Exception:
        pass

set_padded_bounds(ax, region_map, pad_frac=NATIONAL_PAD)
ax.set_axis_off()
ax.set_aspect("equal")

# Sidebar.
y = 0.972
y = draw_wrapped_block(legend_ax, 0.00, y, "Regional median initial watchlist score", width=TITLE_WRAP, fontsize=28, fontweight="bold", line_gap=1.08)
y -= 0.012
y = draw_wrapped_block(legend_ax, 0.00, y, "Each region is coloured by its median ward-level initial watchlist score. Labels on the map show the regional median.", width=SUBTITLE_WRAP, fontsize=14.0, color=TEXT_MID, line_gap=1.20)
y -= 0.035
y = draw_wrapped_block(legend_ax, 0.00, y, "Regional medians", width=TITLE_WRAP, fontsize=14.0, fontweight="bold")

y -= 0.004
for _, row in region_summary.sort_values("median", ascending=False).iterrows():
    legend_ax.text(0.00, y, f"{row['analysis_region']}: {row['median']:.2f}  |  Pop: {fmt_int(row['population'])}", ha="left", va="top", fontsize=11.6, color=TEXT_DARK)
    y -= 0.030

legend_y = max(y - 0.020, 0.230)
legend_ax.text(0.00, legend_y, "Median score band", ha="left", va="top", fontsize=12.5, fontweight="bold", color=TEXT_DARK)
legend_y -= 0.038
for band in SCORE_BAND_ORDER:
    rect = Rectangle((0.00, legend_y - 0.024), 0.030, 0.030, facecolor=SCORE_BAND_COLORS[band], edgecolor=BASE_EDGE, linewidth=0.7)
    legend_ax.add_patch(rect)
    legend_ax.text(0.050, legend_y, band, ha="left", va="top", fontsize=11.2, color=TEXT_DARK)
    legend_y -= 0.043

save_map(fig, "map_20_regional_median_initial_watchlist_score_v2.png", "Region-level map coloured by median initial watchlist score, with population shown in the sidebar.")


Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_20_regional_median_initial_watchlist_score_v2.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_20_regional_median_initial_watchlist_score_v2.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_20_regional_median_initial_watchlist_score_v2.pdf


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/maps/map_20_regional_median_initial_watchlist_score_v2.png')

## 11. High-score concentration map

This map suppresses lower-scoring wards and highlights only wards above a chosen threshold. It helps show whether high scores are regionally concentrated or scattered.

In [18]:
HIGH_SCORE_THRESHOLD = 70

high_gdf = map_gdf[map_gdf["score_for_map"] >= HIGH_SCORE_THRESHOLD].copy()
print(f"Wards scoring {HIGH_SCORE_THRESHOLD}+:", len(high_gdf))
print(high_gdf.groupby("analysis_region").size().sort_values(ascending=False))

fig = plt.figure(figsize=(26.0, 17.0))
fig.patch.set_facecolor(BACKGROUND)
gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[2.15, 5.20], wspace=0.025)
legend_ax = fig.add_subplot(gs[0, 0])
ax = fig.add_subplot(gs[0, 1])
legend_ax.set_facecolor(BACKGROUND)
ax.set_facecolor(BACKGROUND)
legend_ax.set_axis_off()
legend_ax.set_xlim(0, 1)
legend_ax.set_ylim(0, 1)

map_gdf.plot(ax=ax, color="#EFEFEF", edgecolor="none", linewidth=0)
region_geom.boundary.plot(ax=ax, color="#BDBDBD", linewidth=0.50)

# Highlight threshold wards. Use no ward edges to keep the national pattern clear.
high_gdf.plot(ax=ax, color=SCORE_BAND_COLORS["70+"], edgecolor="none", linewidth=0)

if SHOW_SDP_BORDERS and "sdp_contested" in map_gdf.columns:
    sdp_sub = map_gdf[map_gdf["sdp_contested"].fillna(False).astype(bool)]
    if len(sdp_sub):
        sdp_sub.boundary.plot(ax=ax, color=SDP_BORDER_COLOR, linewidth=0.55, alpha=0.85)

set_padded_bounds(ax, map_gdf, pad_frac=NATIONAL_PAD)
ax.set_axis_off()
ax.set_aspect("equal")

# Sidebar.
y = 0.972
y = draw_wrapped_block(legend_ax, 0.00, y, f"High-score ward concentration: {HIGH_SCORE_THRESHOLD}+", width=TITLE_WRAP, fontsize=28, fontweight="bold", line_gap=1.08)
y -= 0.012
y = draw_wrapped_block(legend_ax, 0.00, y, "Grey wards fall below the threshold. Highlighted wards are structural opportunities under the current model, not predicted wins.", width=SUBTITLE_WRAP, fontsize=14.0, color=TEXT_MID, line_gap=1.20)
y -= 0.035

total_pop = high_gdf["population_for_map"].sum(min_count=1)
y = draw_wrapped_block(legend_ax, 0.00, y, f"Wards at {HIGH_SCORE_THRESHOLD}+: {len(high_gdf):,}\nPopulation: {fmt_int(total_pop)}", width=SUBTITLE_WRAP, fontsize=14.0, fontweight="bold")
y -= 0.012

y = draw_wrapped_block(legend_ax, 0.00, y, "By region", width=SUBTITLE_WRAP, fontsize=13.4, fontweight="bold")
y -= 0.004
high_counts = (
    high_gdf.groupby("analysis_region")
    .agg(wards=("WD25CD", "count"), population=("population_for_map", "sum"))
    .reset_index()
    .sort_values(["wards", "population"], ascending=False)
)
for _, row in high_counts.iterrows():
    legend_ax.text(0.00, y, f"{row['analysis_region']}: {int(row['wards']):,} | Pop: {fmt_int(row['population'])}", ha="left", va="top", fontsize=11.4, color=TEXT_DARK)
    y -= 0.030

legend_y = max(y - 0.025, 0.165)
legend_ax.text(0.00, legend_y, "Highlight key", ha="left", va="top", fontsize=12.5, fontweight="bold", color=TEXT_DARK)
rect = Rectangle((0.00, legend_y - 0.060), 0.030, 0.030, facecolor=SCORE_BAND_COLORS["70+"], edgecolor=BASE_EDGE, linewidth=0.7)
legend_ax.add_patch(rect)
legend_ax.text(0.050, legend_y - 0.032, f"{HIGH_SCORE_THRESHOLD}+ wards", ha="left", va="top", fontsize=11.4, color=TEXT_DARK)

if SHOW_SDP_BORDERS:
    rect2 = Rectangle((0.00, legend_y - 0.105), 0.030, 0.030, facecolor="none", edgecolor=SDP_BORDER_COLOR, linewidth=1.5)
    legend_ax.add_patch(rect2)
    legend_ax.text(0.050, legend_y - 0.077, "SDP previously stood", ha="left", va="top", fontsize=11.4, color=TEXT_DARK)

save_map(fig, f"map_30_high_score_{HIGH_SCORE_THRESHOLD}_plus_wards_v2.png", f"England and Wales concentration map for wards scoring {HIGH_SCORE_THRESHOLD}+ with population and SDP-contested borders.")

high_score_summary = (
    high_gdf.groupby("analysis_region")
    .agg(
        high_score_wards=("WD25CD", "count"),
        high_score_population=("population_for_map", "sum"),
        mean_high_score=("score_for_map", "mean"),
        max_high_score=("score_for_map", "max"),
        sdp_contested_high_score_wards=("sdp_contested", "sum"),
    )
    .reset_index()
    .sort_values(["high_score_wards", "mean_high_score"], ascending=False)
)
high_score_summary["high_score_population"] = high_score_summary["high_score_population"].round(0).astype("Int64")
high_score_summary["mean_high_score"] = high_score_summary["mean_high_score"].round(2)
high_score_summary["max_high_score"] = high_score_summary["max_high_score"].round(2)
high_score_summary["sdp_contested_high_score_wards"] = high_score_summary["sdp_contested_high_score_wards"].astype(int)
display(high_score_summary)
save_table(high_score_summary, f"national_high_score_{HIGH_SCORE_THRESHOLD}_plus_summary_v2.csv", f"Summary of wards scoring {HIGH_SCORE_THRESHOLD}+ by region, with population and SDP-contested ward counts.")


Wards scoring 70+: 532
analysis_region
South West                  115
East Midlands                96
Wales                        65
West Midlands                63
South East                   55
East of England              53
North East                   40
North West                   26
Yorkshire and The Humber     19
dtype: int64
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_30_high_score_70_plus_wards_v2.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_30_high_score_70_plus_wards_v2.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\map_30_high_score_70_plus_wards_v2.pdf


,analysis_region,high_score_wards,high_score_population,mean_high_score,max_high_score,sdp_contested_high_score_wards
5,South West,115,725021,73.49,81.99,0
0,East Midlands,96,502624,73.42,81.41,0
6,Wales,65,241957,73.57,81.14,0
7,West Midlands,63,394823,72.43,80.28,0
4,South East,55,368313,73.38,81.00,1
1,East of England,53,373972,72.54,76.73,0
2,North East,40,316309,73.51,81.10,3
3,North West,26,224648,72.86,79.76,0
8,Yorkshire and The Humber,19,225475,73.18,78.63,2


Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\tables\national_high_score_70_plus_summary_v2.csv


WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/tables/national_high_score_70_plus_summary_v2.csv')

## 12. Optional diagnostic charts

These are not replacements for maps. They help check whether the visual pattern matches the underlying distribution.

In [19]:
# Median bar chart by region.
fig, ax = plt.subplots(figsize=(10.5, 6.0))
ordered = region_summary.sort_values("median", ascending=True)
ax.barh(ordered["analysis_region"], ordered["median"])
ax.set_xlabel("Median initial watchlist score")
ax.set_title("Median initial watchlist score by region")
ax.grid(axis="x", linewidth=0.3, alpha=0.5)
fig.tight_layout()
save_map(fig, "chart_01_region_median_initial_watchlist_score_v1.png", "Horizontal bar chart of regional median initial watchlist scores.", dpi=200)

# Boxplot by region.
plot_df = model[["analysis_region", "score_for_map"]].dropna().copy()
region_order = region_summary.sort_values("median", ascending=False)["analysis_region"].tolist()
box_data = [plot_df.loc[plot_df["analysis_region"].eq(r), "score_for_map"].values for r in region_order]

fig, ax = plt.subplots(figsize=(11.5, 6.2))
ax.boxplot(box_data, labels=region_order, vert=False, showfliers=False)
ax.set_xlabel("Initial watchlist score")
ax.set_title("Initial watchlist score distribution by region")
ax.grid(axis="x", linewidth=0.3, alpha=0.5)
fig.tight_layout()
save_map(fig, "chart_02_region_score_distribution_boxplot_v1.png", "Boxplot of ward-level initial watchlist scores by region.", dpi=200)

Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\chart_01_region_median_initial_watchlist_score_v1.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\chart_01_region_median_initial_watchlist_score_v1.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\chart_01_region_median_initial_watchlist_score_v1.pdf
Saved map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\chart_02_region_score_distribution_boxplot_v1.png
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\maps\chart_02_region_score_distribution_boxplot_v1.svg
Saved vector map: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution

WindowsPath('c:/Users/keena/Documents/Electoral_Tribes/data/processed/national_score_distribution_maps_v2_print_quality/maps/chart_02_region_score_distribution_boxplot_v1.png')

## 13. Export manifest

In [20]:
manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "national_score_distribution_maps_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path)
display(manifest)

Saved manifest: c:\Users\keena\Documents\Electoral_Tribes\data\processed\national_score_distribution_maps_v2_print_quality\manifest\national_score_distribution_maps_manifest_v1.csv


,filename,asset_type,description,path,created_at
0,national_region_score_summary_v2.csv,table_csv,Regional summary of initial watchlist score wi...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:25:40
1,national_region_score_band_counts_v1.csv,table_csv,Score-band counts and shares by region.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:25:40
2,national_top_wards_by_region_v1.csv,table_csv,Top 25 wards by initial watchlist score within...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:25:40
3,national_score_map_ward_index_v2.csv,table_csv,"Ward-level map index after joining scores, pop...",c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:26:04
4,map_01_england_wales_initial_watchlist_score_b...,map_png,England and Wales ward-level initial watchlist...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:26:41
5,map_01_england_wales_initial_watchlist_score_b...,map_svg,England and Wales ward-level initial watchlist...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:26:47
6,map_01_england_wales_initial_watchlist_score_b...,map_pdf,England and Wales ward-level initial watchlist...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:26:55
7,map_02_country_england_initial_watchlist_score...,map_png,England ward-level initial watchlist score map...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:27:28
8,map_02_country_england_initial_watchlist_score...,map_svg,England ward-level initial watchlist score map...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:27:34
9,map_02_country_england_initial_watchlist_score...,map_pdf,England ward-level initial watchlist score map...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-06-03T11:27:40


## 14. Expected outputs

Maps are saved to:

```text
data/processed/national_score_distribution_maps_v2_print_quality/maps/
```

Tables are saved to:

```text
data/processed/national_score_distribution_maps_v2_print_quality/tables/
```

With `EXPORT_VECTOR = True`, each map is also exported as SVG and PDF. These are preferable for print and deep zoom. If vector export is too slow or produces very large files, set `EXPORT_VECTOR = False` and use the high-DPI PNGs instead.
